# Phase 1
Training the teacher to avoid obstacles and get to the goal efficiently/quickly. I will likely convert this to python so that I can import it into another file where I run phase 1, then phase 2 and 3, rather than having to run each separately, but I'm not sure yet. 

In [1]:
%cd ..
# need to be able to see environment
# this isn't working for some reason

/home/charlotte/gymnasium-rl/gridworld_v1


In [2]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install tqdm
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
os.listdir()
# os.chdir("GitHub/gymnasium-rl/gridworld_v1")

['.venv',
 'agents',
 'training',
 'checkpoint_models',
 'environment',
 'requirements.txt',
 'all_models',
 '.DS_Store']

In [5]:
# imports
from environment.compass_nav_world import GridWorldBase, TeacherWrapper
from agents.teacher import TeacherAgent, ExperienceReplay
import tqdm
import torch
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
import random
import re
import glob
import os
import sys

In [6]:
mode = "train" # default
model_folder_path = None # for loading a model if in test
# initialization
learning_rate = 0.001
initial_epsilon = 1
epsilon_decay = 0.9995
final_epsilon = 0.05
discount_factor = 0.99

episodes = 10000
max_steps_multiplier = 4
target_update_freq = 500 # for target CNN, in steps
visibility = 4

goal_reward = 10
step_penalty = -0.5

num_special_regions = 1
special_region_rewards = [-5.0]
num_directions = 16 # this is the default anyways
reward_shaping = False # this is also the default

experience_capacity = 8000
batch_size = 64

spawn_widths = [5, 10, 15] # matters for target placement

num_filters_first_layer = 16
final_conv_filters = num_filters_first_layer * 2
target_spatial_size = 1

compass_penalty_multiplier = 0.05 # original

changes = None # will be overriden by papermill if running headlessly, and if not I'll get from input
notes = None

In [7]:
# Parameters
changes = "even lower discount factor :')"
mode = "train"
compass_penalty_multiplier = 0.05
model_folder_path = None
learning_rate = 0.001
epsilon_decay = 0.999
final_epsilon = 0.05
discount_factor = 0.9
episodes = 20000
max_steps_multiplier = 5
target_update_freq = 500
visibility = 4
goal_reward = 10
step_penalty = -0.1
num_special_regions = 1
special_region_rewards = [-5.0]
experience_capacity = 8000
batch_size = 64
spawn_widths = [5, 10, 15, 20]
num_filters_first_layer = 16
final_conv_filters = 32
target_spatial_size = 3
notes = "Ran in the background, may add notes later if this run was important/a checkpoint"


In [8]:
if changes is None: 
    changes = input("What did you change for this round? ")

In [9]:
# initialization
agent = TeacherAgent(num_special_regions, num_directions, learning_rate, initial_epsilon, epsilon_decay, final_epsilon, num_filters_first_layer, final_conv_filters, target_spatial_size, target_update_freq, discount_factor)

experience_replays = {
    w: ExperienceReplay(capacity=experience_capacity, batch_size=batch_size) for w in spawn_widths
}

Using cuda device


In [10]:
if mode == "train": 
    # training
    episode_total_rewards = []
    losses = []
    lengths = defaultdict(list)

    is_headless = not sys.stdout.isatty()
    pbar = tqdm.tqdm(range(episodes), desc="Training", miniters=500 if is_headless else 1, mininterval=60 if is_headless else 0.1)

    for episode in pbar:
        spawn_width = random.choice(spawn_widths)

        max_steps = spawn_width * max_steps_multiplier # scaling up
        base_env = GridWorldBase(num_special_regions, goal_reward, step_penalty, spawn_width, num_directions, reward_shaping, compass_penalty_multiplier)
        env = TeacherWrapper(base_env, visibility, max_steps, special_region_rewards)
        replay = experience_replays[spawn_width]

        episode_losses = [] 
        obs, info = env.reset()
        state = base_env.make_one_agent_grid_relative("teacher", visibility)
        episode_reward = 0

        for step in range(max_steps): 
            action = agent.get_action(state)
            obs, reward, terminated, truncated, info = env.step(action)
            next_state = base_env.make_one_agent_grid_relative("teacher", visibility)

            state_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(agent.device)

            replay.add_experience(state, action, reward, next_state, terminated or truncated) # used to just be terminated, but added truncated to stop bleeding btw episodes which happens when done isn't triggered
            
            episode_reward += reward
            state = next_state

            if replay.can_provide_sample() and step % 4 == 0: # don't learn every step (improved performance I think)
                experiences = replay.sample_batch()
                loss = agent.learn(experiences)
                episode_losses.append(loss)
            

            if terminated or truncated: 
                lengths[spawn_width].append(step + 1) # this is very variable depending on grid size, so I'm storing with the grid size as a key
                break

        avg_loss = sum(episode_losses) / len(episode_losses) if episode_losses else 0
        pbar.set_postfix(epsilon=f"{agent.epsilon:.3f}", reward=f"{episode_reward:.1f}", steps=step + 1, loss=f"{avg_loss:.3f}", refresh=False)

        if episode % 500 == 0 and len(episode_total_rewards) >= 100:
            avg = sum(episode_total_rewards[-100:]) / 100
            print(f"ep {episode} | avg_reward(100)={avg:.1f} | epsilon={agent.epsilon:.3f}", flush=True)
        
        losses.extend(episode_losses)
        agent.decay_epsilon()
        
        episode_total_rewards.append(episode_reward)

    pbar.close()

Training:   0%|                                                                                                                                          | 0/20000 [00:00<?, ?it/s]

Training:   0%|                                                                                       | 0/20000 [00:20<?, ?it/s, epsilon=0.675, loss=0.055, reward=-29.5, steps=50]

ep 500 | avg_reward(100)=-32.3 | epsilon=0.606


ep 1000 | avg_reward(100)=-12.9 | epsilon=0.368


Training:   6%|████▊                                                                       | 1280/20000 [01:00<14:37, 21.32it/s, epsilon=0.278, loss=0.053, reward=-10.6, steps=60]

...

ep 18500 | avg_reward(100)=-0.9 | epsilon=0.050


Training:  94%|███████████████████████████████████████████████████████████████████████▎    | 18781/20000 [14:00<00:53, 22.82it/s, epsilon=0.050, loss=0.063, reward=-5.0, steps=50]

ep 19000 | avg_reward(100)=-2.3 | epsilon=0.050


ep 19500 | avg_reward(100)=-0.2 | epsilon=0.050


Training: 100%|██████████████████████████████████████████████████████████████████████████████| 20000/20000 [14:59<00:00, 22.24it/s, epsilon=0.050, loss=0.017, reward=4.9, steps=3]

In [11]:
if mode == "train": 
    # works no matter what random directory it ends up running from
    if os.path.exists("../all_models"):
        base_dir = "../all_models"
    elif os.path.exists("all_models"):
        base_dir = "all_models"
    else:
        raise FileNotFoundError("Could not find all_models")

    dirs = glob.glob(os.path.join(base_dir, "tm_*"))
    highest_num = 0
    for d in dirs: 
        number = re.search(r'tm_(\d+)$', d)
        if number: 
            highest_num = max(highest_num, int(number.group(1)))

    next_num = highest_num + 1
    run_dir = os.path.join(base_dir, f"tm_{next_num}")
    os.makedirs(run_dir, exist_ok=True)

    hyperparameters = { # for later, in human readable
        "spawn_widths": spawn_widths, 
        "learning_rate": learning_rate, 
        "initial_epsilon": initial_epsilon, 
        "epsilon_decay": epsilon_decay, 
        "final_epsilon": final_epsilon, 
        "discount_factor": discount_factor, 
        "episodes": episodes, 
        "max_steps": max_steps, 
        "target_update_freq": target_update_freq, 
        "visibility": visibility, 
        "goal_reward": goal_reward, 
        "num_special_regions": num_special_regions, 
        "special_region_rewards": special_region_rewards, 
        "experience_capacity": experience_capacity, 
        "batch_size": batch_size, 
        "num_filters_first_layer": num_filters_first_layer, 
        "final_conv_filters": final_conv_filters, 
        "target_spatial_size": target_spatial_size, 
        "step_penalty": step_penalty, 
        "num_directions": num_directions, 
        "reward_shaping": reward_shaping, 
        "compass_penalty_multiplier": compass_penalty_multiplier
    }

    agent.save_model(f'{run_dir}/model_info.pt')

    print(f"base_dir: {base_dir}")
    print(f"run_dir: {run_dir}")

base_dir: all_models
run_dir: all_models/tm_95


In [12]:
if mode == "train": 
    # https://gymnasium.farama.org/introduction/train_agent/
    def get_moving_avgs(arr, window, convolution_mode):
        """Compute moving average to smooth noisy data."""
        return np.convolve(
            np.array(arr).flatten(),
            np.ones(window),
            mode=convolution_mode
        ) / window

    # Smooth over this window
    rolling_length = episodes//20
    fig, axs = plt.subplots(ncols=2, figsize=(12, 5))

    # Episode rewards (win/loss performance)
    axs[0].set_title("Episode rewards")
    reward_moving_average = get_moving_avgs(
        episode_total_rewards,
        rolling_length,
        "valid"
    )
    axs[0].plot(range(len(reward_moving_average)), reward_moving_average)
    axs[0].set_ylabel("Average Reward")
    axs[0].set_xlabel("Episode")


    # Training error (how much we're still learning)
    axs[1].set_title("Training Error")
    training_error_moving_average = get_moving_avgs(
        losses,
        rolling_length,
        "same"
    )
    axs[1].plot(range(len(training_error_moving_average)), training_error_moving_average)
    axs[1].set_ylabel("Temporal Difference Error")
    axs[1].set_xlabel("Step")

    plt.tight_layout()
    plt.savefig(f'{run_dir}/plots.png')
    print(f"Saved to {run_dir}")
    plt.close()

Saved to all_models/tm_95


In [13]:
if mode == "train": 
    rolling_length = episodes // (len(spawn_widths) * 10) # bc split across diff grid sizes
    fig, axs = plt.subplots(ncols=len(spawn_widths), figsize=(12, 5))
    axs = np.atleast_1d(axs) # force to always return an array, even if only 1 length

    for i, key in enumerate(spawn_widths): 
        # Episode lengths (num steps, to reach goal or to truncation)
        axs[i].set_title(f"Episode lengths: {key}")
        if spawn_width not in lengths or len(lengths[key]) < rolling_length: 
            continue # not enough data for grid size
        length_moving_average = get_moving_avgs(
            lengths[key],
            rolling_length,
            "valid"
        )
        axs[i].plot(range(len(length_moving_average)), length_moving_average)
        axs[i].set_ylabel("Average Episode Length")
        axs[i].set_xlabel("Episode")


    plt.tight_layout()
    plt.savefig(f'{run_dir}/length_plots.png')
    print(f"Saved to {run_dir}")
    plt.close()

Saved to all_models/tm_95


In [14]:
if mode == "test": 
   if os.path.exists("../all_models"):
      base_dir = "../all_models"
   elif os.path.exists("all_models"):
      base_dir = "all_models"
   else:
        raise FileNotFoundError("Could not find all_models")

   run_dir = os.path.join(base_dir, f'{model_folder_path}')
   agent.load_model_from_saved(f'{run_dir}/model_info.pt')
   run_dir = model_folder_path

In [15]:
with open(f'{run_dir}/test_result.txt', 'w') as f:
    def run_test(spawn_size, output_file=None):
        base_env = GridWorldBase(num_special_regions, goal_reward, step_penalty, 9, num_directions, reward_shaping, compass_penalty_multiplier)
        env = TeacherWrapper(base_env, visibility, max_steps, special_region_rewards)
        obs, info = env.reset()
        total_reward = 0
        state = base_env.make_one_agent_grid_relative("teacher", visibility)
        went_through_special_region = False
        
        for step in range(max_steps):
            action = agent.get_action(state)
            obs, reward, terminated, truncated, info = env.step(action)
            next_state = base_env.make_one_agent_grid_relative("teacher", visibility)
            if reward in special_region_rewards:
                went_through_special_region = True
            total_reward += reward
            print(f"Step {step} | Action: {action} | Reward: {reward}", file=output_file)
            state = next_state
            if terminated or truncated:
                if truncated:
                    print(f"Truncated after {step+1} steps | Total reward: {total_reward}", file=output_file)
                if terminated:
                    print(f"Reached goal after {step+1} steps | Total reward: {total_reward}", file=output_file)
                if went_through_special_region:
                    print("Went through special region", file=output_file)
                break
    
    for label, size in [("Same size", spawn_width), ("2x size", spawn_width*2)]:
        print(f"\n{label} tests:", file=f)
        for i in range(1, 4):
            print(f"\nTest {i}:\n", file=f)
            run_test(size, output_file=f)

In [16]:
import math
with open(f'{run_dir}/q_value_display.txt', 'w') as f:
    def run_test(world_size, output_file=None):
        agent.epsilon = 0
        base_env = GridWorldBase(num_special_regions, goal_reward, step_penalty, 9, num_directions, reward_shaping, compass_penalty_multiplier)
        env = TeacherWrapper(base_env, visibility, 36, special_region_rewards)
        obs, info = env.reset()
        state = base_env.make_one_agent_grid_relative("teacher", visibility)

        for step in range(36):
            state_tensor = torch.tensor(state, dtype=torch.float32, device=agent.device).unsqueeze(0)
            compass_slice = state_tensor[0, agent.model.num_spatial_channels:, 0, 0].cpu() # not sure if .cpu is necessary but error got mad at me and this is what it suggested
            angle_bin = compass_slice.argmax().item()
            angle = ((angle_bin + 0.5) / base_env.num_directions) * 2 * math.pi # center of bucket (instead of lower edge)

            with torch.no_grad():
                q_values = agent.model(state_tensor)
            action = q_values[0].argmax().item()
            curr_agent_loc = base_env.get_agent_loc_for_name("teacher")
            new_agent_loc = curr_agent_loc + base_env._action_to_direction[action]
            reward = base_env.get_reward_with_shaping(curr_agent_loc, new_agent_loc)
            print(f"step {step} | compass: angle={angle:.3f} | q_values: {q_values[0].tolist()} | action: {action} | reward for action: {reward}", file=output_file)
            
            action = agent.get_action(state)
            obs, reward, terminated, truncated, info = env.step(action)
            state = base_env.make_one_agent_grid_relative("teacher", visibility)
            base_env.render(output_file=output_file)
            
            if terminated or truncated:
                print(f"done in {step+1} steps", file=output_file)
                break
    
    for label, size in [("Same size", spawn_width), ("2x size", spawn_width*2)]:
        print(f"\n{label} tests:", file=f)
        for i in range(1, 4):
            print(f"\nTest {i}:\n", file=f)
            run_test(size, output_file=f)

In [17]:
with open(f'{run_dir}/many_tests_results.txt', 'w') as f:
    for label, size in [("Same size", spawn_widths[len(spawn_widths) // 2]), ("2x size", spawn_width*2), ("3x size", spawn_width*3)]:
        print(f"\n{label} tests:", file=f)
        terminated_count = 0
        truncated_count = 0
        through_special_region_count = 0
        for i in range(1, 50):
            base_env = GridWorldBase(num_special_regions, goal_reward, step_penalty, size, num_directions, reward_shaping, compass_penalty_multiplier)
            env = TeacherWrapper(base_env, visibility, max_steps, special_region_rewards)
            obs, info = env.reset()
            total_reward = 0
            state = base_env.make_one_agent_grid_relative("teacher", visibility)
            
            for step in range(max_steps):
                action = agent.get_action(state)
                obs, reward, terminated, truncated, info = env.step(action)
                next_state = base_env.make_one_agent_grid_relative("teacher", visibility)
                if reward in special_region_rewards:
                    through_special_region_count += 1
                total_reward += reward
                state = next_state
                if terminated or truncated:
                    if truncated: truncated_count += 1
                    if terminated: terminated_count += 1
                    break
        print(f"reached goal: {terminated_count} times", file=f)
        print(f"truncated: {truncated_count} times", file=f)
        print(f"went through special regions: {through_special_region_count} times", file=f)

In [18]:
if mode == "train": 
    # this runs at the end so that I can see graphs/results before writing my note
    human_readable = f""
    for param in hyperparameters: 
        human_readable += f"{str(param)} = {hyperparameters[param]}\n"

    human_readable_time = pbar.format_interval(pbar.format_dict['elapsed'])
    human_readable += f"\n\nTraining Time: {human_readable_time}\n"
    if changes and len(changes) > 0: # recorded before training
        human_readable += "Changes: " + changes + "\n"

    if notes is None: 
        notes = input("Notes on this training run (consider what won't be recorded, like changes to envs or agents): ")

    if notes and len(notes) > 0: 
        human_readable += "Notes: " + notes

    with open(f'{run_dir}/human_info.txt', 'w') as f: 
        f.write(human_readable)